# DS-Fall - 02 | Train DS-Fall-RD A5

Notebook n?y d?ng processed dataset hi?n t?i g?m `bits`, `hifd`, `weda`, `umafall` v? ch?y c?u h?nh ch?nh DS-Fall-RD A5:

- Feature set `tilt12`: raw IMU + `acc_mag`, `gyro_mag`, `jerk`, `roll`, `pitch`, `tilt_delta`.
- Model `ds_fall_rd`: dual-stream encoder, DS-TCN, attention ri?ng cho fall v? direction.
- Direction loss: focal loss c? class weight, ch? ?p d?ng tr?n sample c? `direction_mask=1`.
- `lambda_direction=1.5`, kh?ng augmentation.

Notebook n?y g?i c?ng pipeline v?i script `scripts/run_ds_fall_rd_ablation.py` ?? tr?nh l?ch logic gi?a notebook v? train th?c nghi?m.


In [ ]:
# Mount Google Drive n?u ch?y tr?n Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Google Drive mount skipped:', exc)

import sys
from pathlib import Path

# Ch? c?n ??i PROJECT_ROOT n?u repo n?m ? v? tr? kh?c tr?n Drive.
PROJECT_ROOT = Path('/content/drive/MyDrive/ds-fall')
# Debug local n?u ch?y ngo?i Colab:
# PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

REQ = PROJECT_ROOT / 'requirements.txt'
print('PROJECT_ROOT =', PROJECT_ROOT)
if not REQ.exists():
    print('WARNING: requirements.txt not found. Check PROJECT_ROOT:', PROJECT_ROOT)


In [ ]:
# C?i dependency ch? khi thi?u package ch?nh. Cell n?y an to?n ?? ch?y l?i.
import importlib
import subprocess
MODULE_CHECKS = [('numpy', 'numpy'), ('pandas', 'pandas'), ('scipy', 'scipy'), ('sklearn', 'scikit-learn'), ('matplotlib', 'matplotlib'), ('seaborn', 'seaborn'), ('tensorflow', 'tensorflow')]
def _module_ok(module):
    try:
        importlib.import_module(module)
        return True
    except Exception:
        return False
missing = [pkg for module, pkg in MODULE_CHECKS if not _module_ok(module)]
if missing and REQ.exists():
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)])

import json
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display

from src.config import make_config
from src.utils.io import ensure_dir, load_json
from src.utils.seed import set_seed
from src.experiments.ablation_rd import run_ablation_suite

set_seed(42)
config = make_config(PROJECT_ROOT)
config.enabled_datasets = ('weda', 'bits', 'hifd', 'umafall')
for path in [config.models_dir, config.logs_dir, config.metrics_dir, config.figures_dir / 'training', config.output_dir / 'reports']:
    ensure_dir(path)


In [ ]:
# Ki?m tra TensorFlow v? GPU tr??c khi train.
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs detected:', gpus)
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print('GPU is available and memory growth is enabled.')
    except RuntimeError as exc:
        print('GPU setup error:', exc)
else:
    print('No GPU detected. Training will run on CPU.')


## 1. Check Processed Dataset

Processed data ph?i ???c t?o t? notebook 01 ho?c `scripts/process_datasets.py --datasets weda bits hifd umafall`. Cell n?y x?c nh?n processed set ch? g?m ba dataset ?ang d?ng.


In [ ]:
processed = config.processed_dir
X = np.load(processed / 'X.npy')
y_fall = np.load(processed / 'y_fall.npy')
y_direction = np.load(processed / 'y_direction.npy')
direction_mask = np.load(processed / 'direction_mask.npy')
metadata = pd.read_csv(processed / 'metadata.csv')

assert X.ndim == 3 and X.shape[1:] == (100, 6), f'Unexpected X shape: {X.shape}'
assert len(X) == len(metadata) == len(y_fall) == len(y_direction) == len(direction_mask)
assert np.isfinite(X).all(), 'Processed X contains NaN or Inf.'
expected_datasets = {'bits', 'hifd', 'weda', 'umafall'}
actual_datasets = set(metadata['dataset'].unique())
assert actual_datasets <= expected_datasets, f'Unexpected datasets in processed metadata: {sorted(actual_datasets - expected_datasets)}'

summary = pd.DataFrame([
    {'item': 'X shape', 'value': str(tuple(X.shape))},
    {'item': 'datasets', 'value': ', '.join(sorted(metadata['dataset'].unique()))},
    {'item': 'fall windows', 'value': int(y_fall.sum())},
    {'item': 'non-fall windows', 'value': int((y_fall == 0).sum())},
    {'item': 'direction supervised', 'value': int(direction_mask.sum())},
    {'item': 'resampled windows', 'value': int(metadata['resampled'].astype(bool).sum())},
])
print('Processed data summary')
display(summary)

display(metadata.groupby(['dataset', 'fall_label', 'direction_label']).size().rename('n').reset_index())
display(metadata.groupby(['dataset', 'split']).size().rename('n').reset_index())


## 2. Data Diagnostics

A5 train runner c? th? t? t?o data quality report, direction signal diagnostics v? handcrafted baseline. C?c file ch?nh ???c l?u ? `outputs/reports` v? `outputs/figures`.


In [ ]:
# ??t True n?u ?? ch?y diagnostics ? l?n tr??c v? ch? mu?n train l?i nhanh.
SKIP_DIAGNOSTICS = False
EPOCHS = 80
BATCH_SIZE = 128
LEARNING_RATE = 1e-3

results = run_ablation_suite(
    config,
    ablation_ids=['A5'],
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    processed_x_is_normalized=True,
    skip_diagnostics=SKIP_DIAGNOSTICS,
)

display(results.round(4))


## 3. Review Results

Kh?ng ch?n model theo accuracy t?ng. ?u ti?n: direction macro F1, fall F1, ?? ?n ??nh theo dataset, sau ?? m?i t?i params/latency.


In [ ]:
reports_dir = config.output_dir / 'reports'
ablation_csv = reports_dir / 'ablation_results.csv'
selection_md = reports_dir / 'model_selection.md'

if ablation_csv.exists():
    ablation_results = pd.read_csv(ablation_csv)
    display(ablation_results.round(4))

if selection_md.exists():
    print(selection_md.read_text(encoding='utf-8'))

run_dir = config.output_dir / 'runs' / 'A5_ds_fall_rd_tilt12'
metrics_path = run_dir / 'metrics.json'
if metrics_path.exists():
    metrics = load_json(metrics_path)
    print('A5 fall:', metrics.get('fall'))
    print('A5 direction:', metrics.get('direction'))
    print('A5 per dataset keys:', sorted(metrics.get('per_dataset', {}).keys()))


## 4. Optional TFLite Export

Cell n?y xu?t model A5 t?t nh?t sang TFLite float32 v? dynamic-range quantized. N?u converter kh?ng h? tr? m?i tr??ng hi?n t?i th? b? qua, kh?ng ?nh h??ng model `.keras`.


In [ ]:
try:
    run_dir = config.output_dir / 'runs' / 'A5_ds_fall_rd_tilt12'
    model_path = run_dir / 'model_best.keras'
    if not model_path.exists():
        model_path = run_dir / 'model_final.keras'
    model = tf.keras.models.load_model(model_path, compile=False)

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()
    (config.models_dir / 'ds_fall_rd_a5_float32.tflite').write_bytes(tflite_model)
    print('Saved float32 TFLite:', config.models_dir / 'ds_fall_rd_a5_float32.tflite')

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_quant = converter.convert()
    (config.models_dir / 'ds_fall_rd_a5_dynamic_range.tflite').write_bytes(tflite_quant)
    print('Saved dynamic range quantized TFLite:', config.models_dir / 'ds_fall_rd_a5_dynamic_range.tflite')
except Exception as exc:
    print('TFLite export skipped:', exc)
